# GeoLab Tutorial: Digitizing Non-White Areas in Orthomosaic Files

Drone orthomosaics often contain large white or near-white border areas that represent the region outside the flight coverage. This notebook removes those blank areas and automatically extracts the meaningful content as vector polygons — a workflow equivalent to using QGIS's **Raster → Polygonize** tool, but with fine-grained control over thresholds and geometry simplification.

**Output:**
- A cleaned GeoTIFF (`*_Clean.tif`) with white pixels replaced by black (value = 0)
- An ESRI Shapefile (`.shp`) containing polygons of all non-white areas, filtered by minimum/maximum area and simplified geometry
- Before/after visualization maps rendered inline

> **Target audience:** GIS/QGIS users who work with drone orthomosaics and want to automate boundary digitization using Python.

## Requirements

Install the required packages before running this notebook:

```bash
pip install rasterio geopandas shapely matplotlib
```

Or using conda (recommended on Windows for rasterio):

```bash
conda install -c conda-forge rasterio geopandas shapely matplotlib
```

| Package | Purpose |
|---|---|
| `rasterio` | Reading and writing GeoTIFF files, vectorizing raster masks |
| `geopandas` | Storing, filtering, and exporting polygon geometries as Shapefiles |
| `shapely` | Geometry objects and spatial operations (area, simplification) |
| `numpy` | Fast pixel-wise array operations for mask creation |
| `matplotlib` | Rendering before/after comparison plots inline |

## Step 1: Import Libraries and Set Up Logging

In [ ]:
# Enable inline plot rendering inside Jupyter
%matplotlib inline

import os
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

# NumPy: treat multi-band raster data as 3D arrays (bands × rows × cols)
# for fast vectorised comparisons across all pixels at once
import numpy as np

# rasterio: open, read, and write GeoTIFF files while preserving CRS and transform
import rasterio
# rasterio.features.shapes: convert a binary mask array → list of GeoJSON-like polygon geometries
from rasterio.features import shapes

# geopandas: like pandas but for spatial data — stores geometries alongside attribute rows
import geopandas as gpd

# shapely.geometry.shape: converts a raw GeoJSON dict into a Shapely geometry object
from shapely.geometry import shape

import matplotlib.pyplot as plt

# Configure structured logging so progress messages include timestamps and log levels.
# This is more informative than plain print() statements, especially in long-running jobs.
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("Libraries imported successfully.")

## Step 2: Configuration and Validation

All tunable parameters are collected in a single `ProcessingConfig` dataclass. Centralising settings here makes it easy to adjust thresholds without hunting through the code.

Key parameters to understand:

| Parameter | What it controls |
|---|---|
| `white_threshold` | Pixels with all RGB values ≥ this are treated as "white" (0–255). 250 is a safe default; raise it to 253–255 to only catch pure white. |
| `min_area` | Polygons smaller than this (in map units²) are discarded — removes noise and tiny specks. |
| `max_area` | Polygons larger than this are discarded. Set to `None` for no upper limit. |
| `simplify_tolerance` | Controls how aggressively polygon edges are simplified (in pixels). Higher values = smoother, less detailed boundaries. |
| `connectivity` | 4 = only cardinal neighbours; 8 = diagonals included (produces smoother, more connected regions). |

In [ ]:
@dataclass
class ProcessingConfig:
    """
    All configurable parameters for the orthomosaic white-area removal workflow.

    Values can be overridden via environment variables (for automated pipelines)
    or directly in code when working interactively in this notebook.
    """
    # Pixels with all channel values >= this threshold are considered "white".
    # Range: 0–255. Default 250 catches near-white as well as pure white.
    white_threshold: int = int(os.getenv('WHITE_THRESHOLD', 250))

    # Only keep polygons with area >= min_area (in the CRS's map units, usually m²).
    # Filters out noise and tiny isolated clusters of non-white pixels.
    min_area: float = float(os.getenv('MIN_AREA', 100))

    # Discard polygons larger than max_area. None = no upper limit.
    max_area: Optional[float] = None if not os.getenv('MAX_AREA') else float(os.getenv('MAX_AREA'))

    # Simplification factor (multiplied by pixel size). Higher = smoother borders.
    simplify_tolerance: float = float(os.getenv('SIMPLIFY_TOL', 2.0))

    # Pixel connectivity: 4 = straight edges only, 8 = include diagonals (recommended).
    connectivity: int = int(os.getenv('CONNECTIVITY', 8))

    # The pixel value used to overwrite white areas in the cleaned output raster.
    # 0 = black; set to src.nodata if you want to mark them as NoData instead.
    replacement_value: int = int(os.getenv('REPLACEMENT_VALUE', 0))


def validate_inputs(tif_path: str, output_path: str, config: ProcessingConfig) -> None:
    """
    Validate that the input file exists and all configuration parameters are sensible
    before starting the (potentially slow) processing pipeline.

    Raises FileNotFoundError or ValueError with a descriptive message if any
    check fails, so the user gets clear feedback rather than a cryptic traceback.
    """
    if not Path(tif_path).exists():
        raise FileNotFoundError(f"Input file not found: {tif_path}")

    if not 0 <= config.white_threshold <= 255:
        raise ValueError("white_threshold must be between 0 and 255")

    if config.min_area <= 0:
        raise ValueError("min_area must be positive")

    if config.max_area and config.max_area <= config.min_area:
        raise ValueError("max_area must be greater than min_area")

    # Create the output directory if it doesn't already exist.
    # parents=True creates any intermediate directories as well.
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

## Step 3: Core Processing Functions

Two main functions do the heavy lifting:

1. **`eliminate_white_areas_and_export_tif`** — builds a boolean mask of white pixels, replaces them with `replacement_value`, and writes a cleaned GeoTIFF. Handles 1-band (grayscale), 3-band (RGB), and 4-band (RGBA) images automatically.

2. **`extract_non_white_areas`** — inverts the mask to find *non-white* pixels, converts connected regions to polygon geometries using `rasterio.features.shapes`, filters by area, simplifies edges, and saves to a Shapefile.

In [ ]:
def eliminate_white_areas_and_export_tif(
    tif_path: str,
    output_tif_path: str,
    config: ProcessingConfig
) -> np.ndarray:
    """
    Replace white (or near-white) pixels in an orthomosaic with a fill value and
    save the result as a new GeoTIFF.

    The function reads all bands into memory at once, builds a 2D white-pixel mask,
    overwrites matching pixels with `config.replacement_value`, then writes the
    processed array back while preserving the original CRS, transform, and metadata.
    LZW compression and tiled layout are applied to keep the output file size small.

    Parameters
    ----------
    tif_path : str
        Path to the input orthomosaic GeoTIFF.
    output_tif_path : str
        Path where the cleaned GeoTIFF will be written.
    config : ProcessingConfig
        Processing parameters (white_threshold, replacement_value, etc.).

    Returns
    -------
    np.ndarray
        The processed raster array (bands × rows × cols) after white removal,
        useful for in-memory inspection or downstream processing.
    """
    logger.info(f"Eliminating white areas from: {tif_path}")

    with rasterio.open(tif_path) as src:
        # Read all bands into a single array with shape (bands, rows, cols).
        # Values are kept as-is (no scaling) to preserve the original pixel values.
        ortho_data = src.read()
        processed_data = ortho_data.copy()  # work on a copy so we don't mutate the original

        # --- Build the white-pixel mask ---
        # A pixel is "white" if ALL its spectral channels are >= white_threshold.
        # np.all(..., axis=0) reduces along the band axis, giving a 2D boolean mask.
        if src.count == 4:       # RGBA: compare only the RGB bands, ignore the alpha channel
            white_mask = np.all(ortho_data[:3] >= config.white_threshold, axis=0)
        elif src.count == 3:     # RGB: compare all three bands
            white_mask = np.all(ortho_data >= config.white_threshold, axis=0)
        elif src.count == 1:     # Grayscale: single band comparison
            white_mask = ortho_data[0] >= config.white_threshold
        else:
            # Unusual band count — fall back to inspecting the first three bands
            logger.warning(f"Unusual number of bands ({src.count}), using first 3 for white detection")
            white_mask = np.all(ortho_data[:3] >= config.white_threshold, axis=0)

        # Exclude pixels that are already tagged as NoData in the raster metadata.
        # Without this, NoData pixels might be incorrectly included in the white mask.
        if src.nodata is not None:
            nodata_mask = np.any(ortho_data == src.nodata, axis=0)
            white_mask = white_mask & ~nodata_mask

        # Report how many pixels were detected as white for transparency
        white_count = int(np.sum(white_mask))
        logger.info(
            f"Found {white_count:,} white pixels "
            f"({white_count / white_mask.size * 100:.2f}% of image)"
        )

        # --- Replace white pixels with the fill value ---
        if src.count == 1:
            processed_data[0][white_mask] = config.replacement_value
        else:
            # Set the fill value across every band at the white pixel locations
            for band_idx in range(src.count):
                processed_data[band_idx][white_mask] = config.replacement_value

        # --- Write the cleaned raster ---
        # Copy all metadata from the source (CRS, transform, dtype, nodata) and
        # add LZW compression + tiling to reduce the output file size.
        out_meta = src.meta.copy()
        out_meta.update({
            "compress": "lzw",     # lossless compression — safe for analysis
            "tiled": True,         # tiling enables efficient partial reads in QGIS
            "blockxsize": 512,     # tile width in pixels
            "blockysize": 512,     # tile height in pixels
        })

        with rasterio.open(output_tif_path, 'w', **out_meta) as dst:
            dst.write(processed_data)

        logger.info(f"\u2705 Processed TIFF saved to: {output_tif_path}")
        return processed_data


def extract_non_white_areas(
    tif_path: str,
    output_shp_path: str,
    config: ProcessingConfig
) -> gpd.GeoDataFrame:
    """
    Vectorize the non-white regions of an orthomosaic into a filtered Shapefile.

    Steps:
        1. Build a non-white mask (inverse of the white mask)
        2. Use rasterio.features.shapes() to trace polygon boundaries around
           connected groups of non-white pixels
        3. Filter polygons by min/max area
        4. Optionally simplify polygon edges to reduce vertex count
        5. Save the result as an ESRI Shapefile with an area_m2 attribute

    Parameters
    ----------
    tif_path : str
        Path to the input orthomosaic (can be the original or the cleaned version).
    output_shp_path : str
        Path for the output Shapefile.
    config : ProcessingConfig
        Processing parameters.

    Returns
    -------
    gpd.GeoDataFrame
        GeoDataFrame containing the extracted polygons with an `area_m2` column.
    """
    validate_inputs(tif_path, output_shp_path, config)

    with rasterio.open(tif_path) as src:
        ortho_data = src.read()

        # Build the non-white mask — the inverse logic of eliminate_white_areas.
        # A pixel is "non-white" if ANY of its channels is below the threshold.
        if src.count == 4:
            non_white_mask = np.any(ortho_data[:3] < config.white_threshold, axis=0)
        elif src.count == 3:
            non_white_mask = np.any(ortho_data < config.white_threshold, axis=0)
        elif src.count == 1:
            non_white_mask = ortho_data[0] < config.white_threshold
        else:
            non_white_mask = np.any(ortho_data[:3] < config.white_threshold, axis=0)

        # Exclude NoData pixels from the vectorization mask
        if src.nodata is not None:
            non_white_mask = non_white_mask & ~np.any(ortho_data == src.nodata, axis=0)

        # Determine the area thresholds in pixels.
        # Since pixel size is in the CRS units (usually metres), pixel_area = pixel_size²
        pixel_size = max(abs(src.transform.a), abs(src.transform.e))
        pixel_area_m2 = pixel_size ** 2
        min_pixels = config.min_area / pixel_area_m2
        max_pixels = config.max_area / pixel_area_m2 if config.max_area else None

        # rasterio.features.shapes() traces the boundaries of connected pixel groups.
        # It returns (geometry_dict, pixel_value) pairs for each connected region.
        # We pass the mask itself so only non-white regions are traced.
        polygon_generator = shapes(
            non_white_mask.astype(np.uint8),
            transform=src.transform,
            connectivity=config.connectivity,
            mask=non_white_mask,
        )

        # Filter polygons by area and convert GeoJSON dicts to Shapely geometries
        valid_geometries = []
        for geom_dict, pixel_value in polygon_generator:
            if pixel_value == 1:  # value 1 = non-white region (from our uint8 mask)
                polygon = shape(geom_dict)  # Shapely geometry object
                area_in_pixels = polygon.area  # area in pixel units at this stage
                if area_in_pixels >= min_pixels:
                    if max_pixels is None or area_in_pixels <= max_pixels:
                        valid_geometries.append(polygon)

        if not valid_geometries:
            raise ValueError(
                "No polygons found that meet the area criteria. "
                "Try lowering min_area or adjusting white_threshold."
            )

        # Build a GeoDataFrame — attaching the raster's CRS so the shapefile is georeferenced
        output_gdf = gpd.GeoDataFrame({'geometry': valid_geometries}, crs=src.crs)

        # Add an area attribute. At this stage geometry coordinates are in CRS units (m),
        # so .area gives m², and we scale by pixel_area_m2 to get the real-world area.
        output_gdf['area_m2'] = output_gdf.geometry.area * pixel_area_m2

        # Simplify polygon edges to reduce vertex count and smooth jagged pixel boundaries.
        # tolerance is in CRS units (metres), so pixel_size * factor gives a sensible scale.
        if config.simplify_tolerance > 0:
            output_gdf['geometry'] = output_gdf['geometry'].simplify(
                pixel_size * config.simplify_tolerance,
                preserve_topology=True  # prevent self-intersections during simplification
            )

        output_gdf.to_file(output_shp_path, driver='ESRI Shapefile')
        logger.info(f"\u2705 Shapefile saved to: {output_shp_path} ({len(output_gdf)} polygons)")
        return output_gdf

## Step 4: Visualization Functions

Two helper functions for quality control:

- **`create_visualization`** — plots the extracted polygons coloured by area
- **`create_before_after_comparison`** — shows the original image alongside the cleaned version side-by-side (downsampled for speed on large files)

In [ ]:
def create_visualization(polygon_gdf: gpd.GeoDataFrame, title: str = "Non-White Areas Extracted") -> None:
    """
    Plot the extracted polygons coloured by area (m²) using a viridis colour scale.

    Darker colours indicate larger polygons. Red outlines help distinguish
    individual polygons where they are densely packed.

    Parameters
    ----------
    polygon_gdf : gpd.GeoDataFrame
        Output of extract_non_white_areas() — must have an 'area_m2' column.
    title : str
        The plot title displayed above the map.
    """
    fig, ax = plt.subplots(figsize=(12, 10))

    # column='area_m2' colours each polygon by its area for quick visual QA
    polygon_gdf.plot(
        ax=ax,
        column='area_m2',
        cmap='viridis',
        edgecolor='red',
        alpha=0.7,
        legend=True,
        legend_kwds={'label': 'Area (m²)'}
    )
    plt.title(title, fontsize=16, fontweight='bold')
    plt.xlabel('Easting')
    plt.ylabel('Northing')
    plt.tight_layout()
    plt.show()


def create_before_after_comparison(
    original_tif: str,
    processed_tif: str,
    sample_size: int = 1000
) -> None:
    """
    Display a side-by-side comparison of the original and cleaned orthomosaic.

    Large files are automatically downsampled to at most `sample_size` × `sample_size`
    pixels for display so the comparison renders quickly without loading the full
    raster into memory.

    Parameters
    ----------
    original_tif : str
        Path to the original (pre-processing) GeoTIFF.
    processed_tif : str
        Path to the cleaned (post-processing) GeoTIFF.
    sample_size : int
        Maximum pixel dimension for display. Default 1000 keeps rendering fast.
    """
    fig, (ax_before, ax_after) = plt.subplots(1, 2, figsize=(15, 7))

    def _plot_thumbnail(raster_path: str, ax, panel_title: str) -> None:
        """Read (and optionally downsample) a raster and display it on the given axes."""
        with rasterio.open(raster_path) as src:
            if src.width > sample_size or src.height > sample_size:
                # Compute how much to downsample so the longest dimension = sample_size
                decimation = max(src.width // sample_size, src.height // sample_size, 1)
                # rasterio's out_shape parameter performs the resampling on read
                pixel_data = src.read(
                    window=rasterio.windows.Window(
                        0, 0,
                        min(src.width, sample_size * decimation),
                        min(src.height, sample_size * decimation)
                    ),
                    out_shape=(src.count, sample_size, sample_size)
                )
            else:
                pixel_data = src.read()  # small image — read everything

            # imshow expects (rows, cols, bands) but rasterio returns (bands, rows, cols)
            if src.count >= 3:
                ax.imshow(np.transpose(pixel_data[:3], (1, 2, 0)))  # RGB
            else:
                ax.imshow(pixel_data[0], cmap='gray')               # Grayscale

            ax.set_title(panel_title, fontsize=14, fontweight='bold')
            ax.axis('off')

    _plot_thumbnail(original_tif, ax_before, 'Original Image')
    _plot_thumbnail(processed_tif, ax_after, 'White Areas Eliminated')

    plt.tight_layout()
    plt.show()

## Step 5: Run the Processing Pipeline

Update the **USER INPUTS** block below with your own file paths and configuration values, then run the cell to process the orthomosaic.

In [ ]:
# ==========================================
# USER INPUTS: Update these paths and values
# ==========================================
INPUT_ORTHOPHOTO_PATH   = r"E:\Dokumen Orthophoto\Data Orthomosaic PRM 2025 - Rifqy\cintamangrove.tif"
OUTPUT_SHAPEFILE_PATH   = r"E:\Dokumen Orthophoto\Data Orthomosaic PRM 2025 - Rifqy\cintamangrove.shp"
OUTPUT_CLEAN_TIFF_PATH  = r"E:\Dokumen Orthophoto\Data Orthomosaic PRM 2025 - Rifqy\cintamangrove_Clean.tif"

processing_config = ProcessingConfig(
    white_threshold=253,    # Pixels with all channels >= 253 are treated as white
    min_area=100,           # Discard polygons smaller than 100 m²
    max_area=None,          # No upper area limit
    simplify_tolerance=2.0, # Smooth polygon edges (2× the pixel size)
    connectivity=8,         # 8-connected: diagonals included → less fragmented polygons
    replacement_value=0,    # Replace white pixels with black (0)
)
# ==========================================

try:
    print("\U0001f5fa\ufe0f  Starting orthomosaic processing pipeline...")

    # --- Phase 1: Clean the raster ---
    # Removes white border pixels and writes the cleaned GeoTIFF to disk.
    print("\U0001f4f8  Eliminating white areas and writing cleaned TIFF...")
    cleaned_array = eliminate_white_areas_and_export_tif(
        INPUT_ORTHOPHOTO_PATH,
        OUTPUT_CLEAN_TIFF_PATH,
        processing_config
    )

    # --- Phase 2: Vectorize non-white regions ---
    # Traces polygon boundaries around the meaningful image content and saves as Shapefile.
    print("\U0001f50d  Extracting non-white area polygons...")
    result_polygons_gdf = extract_non_white_areas(
        INPUT_ORTHOPHOTO_PATH,
        OUTPUT_SHAPEFILE_PATH,
        processing_config
    )

    # --- Summary ---
    total_area_m2 = result_polygons_gdf['area_m2'].sum()
    print(f"\n\u2705 Processing complete!")
    print(f"\U0001f4ca  Polygons found     : {len(result_polygons_gdf)}")
    print(f"\U0001f4ca  Total area         : {total_area_m2:.2f} m² ({total_area_m2 / 10000:.2f} ha)")
    print(f"\U0001f4be  Clean TIFF saved   : {OUTPUT_CLEAN_TIFF_PATH}")
    print(f"\U0001f4be  Shapefile saved    : {OUTPUT_SHAPEFILE_PATH}")

    # --- Phase 3: Render quality-control visualizations ---
    print("\n\U0001f5bc\ufe0f  Rendering visualizations...")
    create_visualization(result_polygons_gdf, "Extracted Non-White Areas")
    create_before_after_comparison(INPUT_ORTHOPHOTO_PATH, OUTPUT_CLEAN_TIFF_PATH)

except FileNotFoundError as e:
    print(f"\u274c  File not found: {e}")
except ValueError as e:
    print(f"\u274c  Configuration error: {e}")
except Exception as e:
    print(f"\u274c  Unexpected error: {e}")
    logger.exception("Detailed traceback:")